In [ ]:

# [own] names the stage expects when there is no base notebook
import json, time
from pathlib import Path
import numpy as np, pandas as pd
NB_T0 = time.time()
WORK = Path('/kaggle/working')
DATA = next(p for p in [Path('/kaggle/input/rsna-knee-abnormality-detection'),
                        Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')] if (p / 'test.csv').is_file())
test_ids = pd.read_csv(DATA / 'test.csv')['StudyInstanceUID'].astype(str).tolist()
cols = list(pd.read_csv(DATA / 'sample_submission.csv', nrows=1).columns)
labels = cols[1:]
n = len(test_ids)
def pct(v):
    return pd.Series(np.asarray(v, dtype=np.float64)).rank(method='average', pct=True).values


In [ ]:
FLEET_V6 = '#!/usr/bin/env python3\n"""RSNA Knee Abnormality Detection: Fast Inference & Efficiency Track Pipeline\nSystematic Visible-Data Teacher Probe, Corpus Parity Auditor & Production Engine\n\nKey architectural features:\n  1. Exact parity with Dread\'s preprocessed corpus:\n     - Canonical 44-slice slot contract: Sagittal FS (12), Sagittal (10), Coronal FS (8), Coronal (6), Axial (8)\n     - Per-slot (per-series) 2% and 98% percentile normalization matching scanner contrasts\n     - Full K_EVAL=94 window center evaluation across the entire 44-slice volume\n  2. Stage-by-stage parity auditor comparing live DICOM decoding against all_vols.npy on 20 studies\n  3. Automated visible-data Teacher Probe validation (58 gold + 300 train studies)\n  4. Diagnostic exception logging (FIRST 5 full tracebacks captured)\n  5. Hard guard: aborts loudly if fallbacks > 5% of studies (prevents 0.500 LB submissions)\n  6. Multi-GPU dual-T4 producer-consumer queue with exact index alignment and ID verification\n"""\nimport os, sys, glob, time, math, queue, threading, traceback, shutil\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nimport numpy as np\nimport pandas as pd\nimport pydicom\nimport cv2\nfrom pydicom.pixel_data_handlers.util import apply_modality_lut\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport timm\n\n# Hardware & CUDA configuration\nos.environ.setdefault("HF_HUB_OFFLINE", "1")\nos.environ.setdefault("TRANSFORMERS_OFFLINE", "1")\ntorch.backends.cudnn.benchmark = True\ntorch.backends.cuda.matmul.allow_tf32 = True\n\nLAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",\n       "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\n\n# Model & Preprocessing Contract (Frontier 96-slice native 384px contract)\nIMG = 384\nCROP_MM = 140.0\nSLOTS = [\n    ("Sagittal", 1, 26),   # Sagittal Fluid-Sensitive (Fat-Suppressed)\n    ("Sagittal", 0, 22),   # Sagittal Non-Fluid-Sensitive (Proton Density / T1 / T2)\n    ("Coronal",  1, 18),   # Coronal Fluid-Sensitive\n    ("Coronal",  0, 12),   # Coronal Non-Fluid-Sensitive\n    ("Axial",   -1, 18),   # Axial (any fluid-sensitivity preference)\n]\nMAXS = sum(k for _, _, k in SLOTS)  # 96 slices\nSLOT_SIZES = [26, 22, 18, 12, 18]\nSAGITTAL_SLOTS = (0, 1)\n_b_slots = np.cumsum([0] + SLOT_SIZES)\n_s_map = np.searchsorted(_b_slots, np.arange(MAXS), side=\'right\') - 1\n\nK_EVAL = 94     # default 96 layout: 94 = MAXS - 2 window centers; re-derived from ckpt slot_sizes in main()\nWIN_CHUNK = 14  # Optimal benchmarked chunk size\n\n# Mode Toggle: Set True for standard LB submissions (Anatomical Mirror TTA gives +0.002 AUC)\nUSE_TTA = os.environ.get("RAPTOR_TTA", "1") != "0"   # RAPTOR_TTA=0 halves GPU time\n\nDICOM_HEADER_TAGS = [\n    \'ImageOrientationPatient\',\n    \'ImagePositionPatient\',\n    \'PixelSpacing\',\n    \'InstanceNumber\'\n]\n\n# Exception diagnostics tracker\n_MAX_EXC_LOGS = 5\n_exc_log_count = 0\n_exc_lock = threading.Lock()\n\ndef record_exception(source, sid, exc):\n    global _exc_log_count\n    with _exc_lock:\n        if _exc_log_count < _MAX_EXC_LOGS:\n            _exc_log_count += 1\n            print(f"\\n{\'!\' * 80}", flush=True)\n            print(f"[DIAGNOSTIC EXCEPTION #{_exc_log_count}] in {source} for study {sid}: {type(exc).__name__}: {exc}", flush=True)\n            traceback.print_exc()\n            print(f"{\'!\' * 80}\\n", flush=True)\n\n\n# ============================================================================\n# Architecture Definition\n# ============================================================================\ndef apply_norm(x, norm):\n    if norm != "imagenet":\n        return x\n    mean = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(3, 1, 1)\n    std = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(3, 1, 1)\n    return (x - mean) / std\n\n\ndef build_backbone(arch, pretrained=False):\n    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))\n    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))\n    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)\n    if is_vit:\n        kw.update(global_pool="token", dynamic_img_size=True)\n    else:\n        kw.update(global_pool="avg")\n    return timm.create_model(arch, **kw)\n\n\nclass RaptorClassifier(nn.Module):\n    """Raptor encoder + per-diagnosis attention-MIL head (12 findings)."""\n    def __init__(self, backbone, F_dim=384, n=12, drop=0.2, slot_emb=False, seq_head=False, pos_emb=False):\n        super().__init__()\n        self.backbone = backbone\n        self.F_dim = F_dim\n        self.slot_emb = nn.Embedding(5, F_dim) if slot_emb else None\n        if self.slot_emb is not None:\n            nn.init.zeros_(self.slot_emb.weight)\n        self.pos_proj = nn.Linear(3, F_dim) if pos_emb else None   # [p, sin(pi p), cos(pi p)] -> F, starts as a no-op\n        if self.pos_proj is not None:\n            nn.init.zeros_(self.pos_proj.weight); nn.init.zeros_(self.pos_proj.bias)\n        self.seq_head = bool(seq_head)\n        self.gru = nn.GRU(F_dim, F_dim // 2, num_layers=2,\n                          batch_first=True, bidirectional=True,\n                          dropout=0.1) if self.seq_head else None\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),\n                                 nn.Linear(256, n))\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.aux_norm = nn.LayerNorm(F_dim) if self.seq_head else None\n        self.aux_att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),\n                                     nn.Linear(256, n)) if self.seq_head else None\n        self.aux_clsW = nn.Parameter(torch.zeros(n, F_dim)) if self.seq_head else None\n        self.aux_clsb = nn.Parameter(torch.zeros(n)) if self.seq_head else None\n        if self.aux_clsW is not None:\n            nn.init.trunc_normal_(self.aux_clsW, std=0.02)\n        self.n = n\n\n    win_chunk = 0   # >0: run the backbone over chunks of this many windows under activation checkpointing (8 GB cards)\n    stkim_k = 0     # >0 (training only): ACMIL stochastic top-K masking of each finding\'s top-k windows\n    stkim_p = 0.7   # probability that each of those top-k windows is hidden\n    seq_tracks = 1  # tiles per window position (2 with zoom: field, close-up); the GRU runs each track on its own\n\n    def _attend(self, h):\n        """Per-finding attention weights (B,K,n). In training with stkim_k > 0 each finding\'s top-k\n        windows are hidden with probability stkim_p (ACMIL, ECCV 2024); eval is unchanged."""\n        a = self.att(h)\n        if self.training and self.stkim_k > 0 and a.shape[1] > self.stkim_k:\n            top = a.topk(self.stkim_k, dim=1).indices\n            hit = torch.zeros_like(a, dtype=torch.bool).scatter_(1, top, True)\n            a = a.masked_fill(hit & (torch.rand_like(a) < self.stkim_p), float("-inf"))\n        return torch.softmax(a, dim=1)\n\n    def _seq(self, feats):\n        """bi-GRU over the windows in slice order. With seq_tracks = T tiles per position (field, close-up,\n        field, ...) each track is its own sequence, so the GRU never alternates between scales."""\n        T = self.seq_tracks\n        if T == 1:\n            return self.gru(feats)[0]\n        B, K, F = feats.shape\n        S = K // T\n        g = feats[:, :S * T].reshape(B, S, T, F).transpose(1, 2).reshape(B * T, S, F)\n        g = self.gru(g)[0].reshape(B, T, S, F).transpose(1, 2).reshape(B, S * T, F)\n        return torch.cat([g, g.new_zeros(B, K - S * T, F)], 1)  # a cut-off last tile gets no GRU term\n\n    def encode(self, x):\n        B, K = x.shape[:2]\n        flat = x.flatten(0, 1)\n        if self.win_chunk > 0 and self.training and torch.is_grad_enabled():\n            from torch.utils.checkpoint import checkpoint\n            f = torch.cat([checkpoint(self.backbone, c, use_reentrant=False) for c in flat.split(self.win_chunk, 0)], 0)\n        elif self.win_chunk > 0:\n            f = torch.cat([self.backbone(c) for c in flat.split(self.win_chunk, 0)], 0)\n        else:\n            f = self.backbone(flat)\n        return f.view(B, K, -1)\n\n    def _add_slots(self, feats, slots):\n        if slots is None:\n            return feats\n        if self.pos_proj is not None and slots.is_floating_point():   # pack_slot_pos: slot id + packed position\n            sid = slots.floor()\n            p = (slots - sid) / 0.4995 - 1.0\n            feats = feats + self.pos_proj(torch.stack([p, torch.sin(math.pi * p), torch.cos(math.pi * p)], -1).to(feats.dtype))\n            slots = sid.long()\n        if self.slot_emb is not None:\n            feats = feats + self.slot_emb(slots)\n        return feats\n\n    def head(self, feats, slots=None):\n        feats = self._add_slots(feats, slots)\n        if self.seq_head:\n            feats = feats + self._seq(feats)\n        h = self.norm(feats)\n        a = self._attend(h)\n        pooled = torch.einsum("bkn,bkf->bnf", a, h)\n        logits = (pooled * self.clsW).sum(-1) + self.clsb\n        return logits\n\n    def aux_head(self, feats, slots=None):\n        """Independent attention-MIL copy on the PRE-GRU features (training aux loss)."""\n        feats = self._add_slots(feats, slots)\n        h = self.aux_norm(feats)\n        a = self.aux_att(h)\n        a = torch.softmax(a, dim=1)\n        pooled = torch.einsum("bkn,bkf->bnf", a, h)\n        logits = (pooled * self.aux_clsW).sum(-1) + self.aux_clsb\n        return logits\n\n    def forward(self, x, slots=None):\n        return self.head(self.encode(x), slots=slots)\n\n    def forward_with_aux(self, x, slots=None):\n        """Returns (main_logits, aux_logits). Eval/inference use main only."""\n        feats = self.encode(x)\n        return self.head(feats, slots=slots), self.aux_head(feats, slots=slots)\n\n\n# ============================================================================\n# Robust Preprocessing & DICOM Decoders (Frontier 96-Slice Native 384px Contract)\n# ============================================================================\ndef _read_header_single(item):\n    sid, f = item if isinstance(item, tuple) else ("unknown", item)\n    x_lps = float("nan")\n    try:\n        h = pydicom.dcmread(f, stop_before_pixels=True, specific_tags=DICOM_HEADER_TAGS, force=True)\n        iop = getattr(h, \'ImageOrientationPatient\', None)\n        ipp = getattr(h, \'ImagePositionPatient\', None)\n        if iop is not None and ipp is not None and len(iop) == 6:\n            r = np.array(iop[:3], float)\n            c = np.array(iop[3:], float)\n            n = np.cross(r, c)\n            pos = float(np.dot(np.array(ipp, float), n))\n            x_lps = float(ipp[0])\n        else:\n            pos = float(getattr(h, \'InstanceNumber\', 0) or 0)\n        ps = getattr(h, \'PixelSpacing\', None)\n        ps = float(ps[0]) if ps is not None else 0.5\n        return pos, f, ps, x_lps\n    except Exception as e:\n        record_exception("_read_header_single", sid, e)\n        return 0.0, f, 0.5, float("nan")\n\n\ndef read_px(item):\n    sid, f = item if isinstance(item, tuple) else ("unknown", item)\n    if f is None:\n        return None\n    try:\n        d = pydicom.dcmread(f, force=True)\n        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)\n        if str(getattr(d, \'PhotometricInterpretation\', \'\')) == \'MONOCHROME1\':\n            a = a.max() - a\n        return a\n    except Exception as e:\n        record_exception("read_px", sid, e)\n        return None\n\n\ndef mm_crop_resize(a, ps):\n    h, w = a.shape\n    cpx = int(round(CROP_MM / max(ps, 1e-3)))\n    cpx = max(1, min(cpx, min(h, w)))\n    y0 = (h - cpx) // 2\n    x0 = (w - cpx) // 2\n    a = a[y0 : y0 + cpx, x0 : x0 + cpx]\n    return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)\n\n\ndef _safe_fluid(val):\n    if val is None or pd.isna(val):\n        return -1\n    try:\n        return int(float(val))\n    except (ValueError, TypeError):\n        return -1\n\n\ndef pick_series_for_slot(rows, plane, fluid, used):\n    plane_norm = str(plane).strip().capitalize()\n    cands = [r for r in rows if str(r.get(\'Anatomical_Plane\', \'\')).strip().capitalize() == plane_norm and str(r.get(\'SeriesInstanceUID\', \'\')) not in used]\n    if fluid in (0, 1):\n        pref = [r for r in cands if _safe_fluid(r.get(\'Fluid_Sensitive\')) == fluid]\n        if pref:\n            return pref[0]\n    return cands[0] if cands else None\n\n\nPICK_SPAN = (0.06, 0.94)   # legacy test-time span (proven LB notebooks); new-layout checkpoints use the corpus span\n\n\ndef _pick_slot_slice_indices(n, k):\n    """Per-slot slice picks: linspace over PICK_SPAN (corpus builder span (0.02, 0.98) for new-layout checkpoints).\n\n    Same rule as build_corpus96_stream.py: duplicates when the series is shorter\n    than the slot (linspace with repeats), [0]*k for a single-slice series.\n    """\n    lo, hi = int(n * PICK_SPAN[0]), int(n * PICK_SPAN[1]) - 1\n    hi = max(hi, lo)\n    return np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k\n\n\ndef build_study_optimized(sid, ser_records, test_dir, io_pool):\n    """Loads and preprocesses a study matching Dread\'s corpus contract bit-for-bit."""\n    t0 = time.time()\n    t_header, t_pixel, t_prep = 0.0, 0.0, 0.0\n\n    rows = ser_records.get(sid, [])\n    used = set()\n    slot_plans = []\n    sag_x = []   # IPP x of sagittal slices: LPS x > 0 = patient\'s left -> left knee (same rule as the training table)\n\n    # Stage 1: Fast header reading & slice sorting across series\n    t_h0 = time.time()\n    for plane, fluid, k in SLOTS:\n        r = pick_series_for_slot(rows, plane, fluid, used)\n        if r is None:\n            slot_plans.append((None, k, 0.5, None))\n            continue\n        used.add(r[\'SeriesInstanceUID\'])\n\n        # Check both nested test_dir/<sid>/<series> and flat test_dir/<series> layouts\n        sdir = os.path.join(test_dir, str(sid), str(r[\'SeriesInstanceUID\']))\n        if not os.path.exists(sdir):\n            sdir = os.path.join(test_dir, str(r[\'SeriesInstanceUID\']))\n        if os.path.isdir(sdir):\n            fs = [os.path.join(sdir, f) for f in os.listdir(sdir) if f.endswith(".dcm")]\n        else:\n            fs = []\n        if not fs:\n            slot_plans.append((None, k, 0.5, r[\'SeriesInstanceUID\']))\n            continue\n\n        header_items = [(sid, f) for f in fs]\n        if len(fs) > 4:\n            recs = list(io_pool.map(_read_header_single, header_items))\n        else:\n            recs = [_read_header_single(item) for item in header_items]\n        recs.sort(key=lambda x: x[0])\n        med_ps = float(np.median([r[2] for r in recs])) if recs else 0.5\n        ordered = [(r[1], r[2]) for r in recs]\n        if plane == "Sagittal":\n            sag_x += [r[3] for r in recs]\n        slot_plans.append((ordered, k, med_ps, r[\'SeriesInstanceUID\']))\n\n    t_header = time.time() - t_h0\n\n    # Stage 2: Pick target slices per slot\n    t_p0 = time.time()\n    # Group slice requests by slot to preserve PER-SLOT normalization parity\n    slot_slice_requests = []  # list of list of (target_idx, fp, ps)\n\n    cur_idx = 0\n    all_files_to_read = []\n    for ordered, k, med_ps, ser_uid in slot_plans:\n        this_slot_requests = []\n        if ordered is None or not ordered:\n            cur_idx += k\n            slot_slice_requests.append(this_slot_requests)\n            continue\n        n = len(ordered)\n        picks = _pick_slot_slice_indices(n, k)\n\n        for p in picks:\n            if cur_idx >= MAXS: break\n            fp, ps = ordered[min(p, n - 1)]\n            this_slot_requests.append((cur_idx, fp, ps if ps > 0 else med_ps))\n            all_files_to_read.append((sid, fp))\n            cur_idx += 1\n        slot_slice_requests.append(this_slot_requests)\n        if cur_idx >= MAXS: break\n\n    # Parallel pixel decompression\n    if len(all_files_to_read) > 4:\n        all_loaded = list(io_pool.map(read_px, all_files_to_read))\n    else:\n        all_loaded = [read_px(item) for item in all_files_to_read]\n    t_pixel = time.time() - t_p0\n\n    # Map loaded pixel arrays back to each slot\n    t_c0 = time.time()\n    vol = np.zeros((MAXS, IMG, IMG), np.uint8)\n    loaded_iter = iter(all_loaded)\n    total_valid_slices = 0\n\n    # Stage 3: Exact Per-Slot Percentile Normalization & mm-Cropping (Dread Parity)\n    for this_slot_requests in slot_slice_requests:\n        slot_arrs = []\n        for req in this_slot_requests:\n            arr = next(loaded_iter)\n            slot_arrs.append(arr)\n\n        valid_in_slot = [a for a in slot_arrs if a is not None]\n        total_valid_slices += len(valid_in_slot)\n\n        if valid_in_slot:\n            allpx = np.concatenate([a.ravel() for a in valid_in_slot])\n            loq, hiq = np.percentile(allpx, [2.0, 98.0])\n        else:\n            loq, hiq = 0.0, 1.0\n\n        for req, arr in zip(this_slot_requests, slot_arrs):\n            target_idx, _, ps = req\n            if target_idx >= MAXS: break\n            if arr is None: continue\n            aw = np.clip((arr - loq) / (hiq - loq + 1e-6), 0, 1)\n            aw = mm_crop_resize(aw, ps)\n            vol[target_idx] = (aw * 255).astype(np.uint8)\n\n    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)\n    t_prep = time.time() - t_c0\n\n    timings = {\n        "header": t_header,\n        "pixel": t_pixel,\n        "prep": t_prep,\n        "total_io": time.time() - t0,\n        "decoded_slices": total_valid_slices,\n        "total_requested": len(all_files_to_read),\n        "side": (1.0 if np.nanmean(sag_x) > 0 else -1.0) if np.isfinite(sag_x).any() else 0.0,\n    }\n    return vol, mask, timings\n\n\n# ============================================================================\n# GPU Inference & Full Window Centers (K_EVAL = 42)\n# ============================================================================\ndef bounds_from_sizes(sizes):\n    """Cumulative 5-slot bounds from per-slot slice counts.\n\n    E.g. [26, 22, 18, 12, 18] -> [(0, 26), (26, 48), (48, 66), (66, 78), (78, 96)].\n    Re-derived at startup from --slot_sizes; any 5-slot layout works.\n    """\n    bounds, start = [], 0\n    for k in sizes:\n        k = int(k)\n        bounds.append((start, start + k))\n        start += k\n    return bounds\n\n\nSLOT_BOUNDS = bounds_from_sizes([26, 22, 18, 12, 18])\n\n\ndef slot_of(idx: int) -> int:\n    """Return slot index (0..4) for a slice index."""\n    for s, (start, end) in enumerate(SLOT_BOUNDS):\n        if start <= idx < end:\n            return s\n    if idx < 0:\n        return 0\n    return len(SLOT_BOUNDS) - 1\n\n\nFIELD_MM = 140.0   # every corpus slice is a 140 mm square field at 384 px (build_corpus96_stream mm_crop_and_resize)\n\n\ndef zoom_px(zoom_mm, img=384):\n    """Side in corpus pixels of a centred close-up of zoom_mm millimetres (0 = off)."""\n    return int(round(img * float(zoom_mm) / FIELD_MM)) if zoom_mm else 0\n\n\ndef window_pos(c: int, mask: np.ndarray, side_sign: float, mirrored: bool) -> float:\n    """Anatomical position of window centre c in [-1, 1] along its slot\'s slice axis (--pos_emb).\n\n    The corpus stores every series in true 3D order (sorted by IPP . slice normal; a 600-series DICOM sample had\n    one direction per plane): sagittal patient-left -> patient-right, coronal front -> back, axial foot -> head.\n    Sagittal becomes lateral (-1) -> medial (+1) via the knee side (+1 left, -1 right, 0 unknown -> 0).\n    A training mirror (mirror_volume) reverses sagittal order, so it flips that sign; coronal/axial flips are\n    in-plane and leave the through-plane position alone.\n    """\n    s = slot_of(c)\n    start, _ = SLOT_BOUNDS[s]\n    n = int(np.sum(mask[start:SLOT_BOUNDS[s][1]] > 0))\n    t = min(max(2.0 * (c - start) / max(n - 1, 1) - 1.0, -1.0), 1.0)\n    if s in (0, 1):\n        t *= side_sign * (-1.0 if mirrored else 1.0)\n    return float(t)\n\n\ndef pack_slot_pos(slot_ids, pos):\n    """--pos_emb rides on the existing slot tensor: integer part = slot id, fraction = (pos + 1) / 2 * 0.999.\n    ponytail: one float tensor instead of a new batch element threaded through every train/eval loop."""\n    return torch.tensor([s + (p + 1.0) * 0.4995 for s, p in zip(slot_ids, pos)], dtype=torch.float32)\n\n\ndef pick_centers(valid, count, train=False, clean_windows=False, mask=None, max_d=None, cap=False, slots=None):\n    """Pick 3-slice window centers.\n\n    If clean_windows is True, a center c is allowed only if c-1, c, c+1 are all real\n    (mask > 0) AND in the same slot (slot_of(c-1) == slot_of(c) == slot_of(c+1)).\n    Otherwise, picks from [lo+1, hi-1] matching legacy behavior.\n    max_d defaults to the full layout depth (sum of slot sizes).\n    """\n    if max_d is None:\n        max_d = sum(end - start for start, end in SLOT_BOUNDS)\n    lo, hi = int(valid.min()), int(valid.max())\n    if clean_windows and mask is not None:\n        cs = [\n            c for c in range(1, max_d - 1)\n            if mask[c - 1] > 0 and mask[c] > 0 and mask[c + 1] > 0\n            and slot_of(c - 1) == slot_of(c) == slot_of(c + 1)\n        ]\n    else:\n        cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]\n    if slots is not None:   # specialist: only windows centred in these slots\n        cs = [c for c in cs if slot_of(c) in slots]\n    if not cs:\n        cs = [max(1, min((lo + hi) // 2, max_d - 2))]\n    if train:\n        reps = count // len(cs) + 1\n        pool = cs * reps\n        random.shuffle(pool)\n        return pool[:count]\n    # eval: evenly spaced deterministic\n    if cap:\n        count = min(count, len(cs))   # fleet inference: never repeat a center\n    idx = np.linspace(0, len(cs) - 1, count).round().astype(int)\n    return [cs[i] for i in idx]\n\n\n# ============================================================================\n# ROI inference (anatomical joint crop + compartment tiling for specialist ckpts)\n# ----------------------------------------------------------------------------\n# _roi_square_box / _roi_compartments: verbatim copies of\n# roi_localize.square_box / roi_localize.compartments (train_knee.py imports\n# these same objects). _roi_slice_box: verbatim per-slice body of\n# build_roi_boxes._process_chunk\'s `for c in range(D)` loop. The Tuple/Optional\n# stand-ins below exist only so the verbatim copies need no new import (this\n# file stays Kaggle self-contained); they are not used at runtime.\n# ============================================================================\nTuple = tuple\n\n\nclass _OptionalAlias:\n    def __class_getitem__(cls, item):\n        return item\n\n\nOptional = _OptionalAlias\n\n\ndef _roi_square_box(\n    sbox: Tuple[int, int, int, int],\n    W: int,\n    H: int,\n    pad: float = 0.06,\n    mode: str = "tight",\n    centroid: Optional[Tuple[float, float]] = None\n) -> Tuple[int, int, int, int]:\n    """Expands a tissue bounding box into a square crop centered on the joint.\n\n    Args:\n        sbox: (x0, y0, x1, y1) bounding box of tissue\n        W: Width of image\n        H: Height of image\n        pad: Fraction of box size to add as padding\n        mode: "tight" (minimal tissue frame), "safe" (conservative 1.2x scale), or "compartment"\n        centroid: (cx, cy) optional center-of-mass to center the square box\n\n    Returns:\n        (x0, y0, x1, y1) integer coordinates clamped to [0, W] and [0, H]\n    """\n    x0, y0, x1, y1 = sbox\n\n    # Handle degenerated or empty boxes\n    if x1 <= x0 or y1 <= y0:\n        return (0, 0, W, H)\n\n    bw = x1 - x0\n    bh = y1 - y0\n\n    # Apply padding\n    pad_px = int(max(bw, bh) * pad)\n    if mode == "safe":\n        pad_px += int(max(bw, bh) * 0.10)\n\n    # Square dimension\n    side = max(bw, bh) + 2 * pad_px\n    side = min(side, min(W, H))\n\n    # Center determination\n    if centroid is not None and centroid[0] > 0 and centroid[1] > 0:\n        cx, cy = centroid\n    else:\n        cx = (x0 + x1) / 2.0\n        cy = (y0 + y1) / 2.0\n\n    # Half side\n    half = side / 2.0\n    new_x0 = int(round(cx - half))\n    new_y0 = int(round(cy - half))\n\n    # Shift if out of bounds\n    if new_x0 < 0:\n        new_x0 = 0\n    elif new_x0 + side > W:\n        new_x0 = W - side\n\n    if new_y0 < 0:\n        new_y0 = 0\n    elif new_y0 + side > H:\n        new_y0 = H - side\n\n    new_x1 = new_x0 + side\n    new_y1 = new_y0 + side\n\n    # Final boundary clamp\n    new_x0 = max(0, min(new_x0, W - 1))\n    new_y0 = max(0, min(new_y0, H - 1))\n    new_x1 = max(new_x0 + 1, min(new_x1, W))\n    new_y1 = max(new_y0 + 1, min(new_y1, H))\n\n    return (new_x0, new_y0, new_x1, new_y1)\n\n\ndef _roi_compartments(\n    sq: Tuple[int, int, int, int],\n    overlap: float = 0.12\n) -> Tuple[Tuple[int, int, int, int], Tuple[int, int, int, int]]:\n    """Splits a square joint box into Medial and Lateral compartment tiles with overlap.\n\n    Args:\n        sq: (x0, y0, x1, y1) square bounding box\n        overlap: Fraction of width to overlap between compartments (default 0.12 = 12%)\n\n    Returns:\n        (left_compartment, right_compartment)\n    """\n    x0, y0, x1, y1 = sq\n    w = x1 - x0\n\n    if w <= 2:\n        return (sq, sq)\n\n    half_w = w / 2.0\n    ov_px = int(round(w * overlap))\n\n    # Left compartment: x0 to middle + overlap\n    left_x1 = int(round(x0 + half_w + ov_px))\n    left_x1 = min(x1, max(x0 + 1, left_x1))\n    left_box = (x0, y0, left_x1, y1)\n\n    # Right compartment: middle - overlap to x1\n    right_x0 = int(round(x0 + half_w - ov_px))\n    right_x0 = max(x0, min(x1 - 1, right_x0))\n    right_box = (right_x0, y0, x1, y1)\n\n    return (left_box, right_box)\n\n\ndef _roi_slice_box(vol, mask, c, sboxes, cens, i_local, W, H, x_grid, y_grid):\n    """Per-slice tissue box + centroid for one slice (writes sboxes/cens rows).\n\n    Verbatim body of build_roi_boxes._process_chunk\'s `for c in range(D)` loop.\n    """\n    if mask[c] == 0:\n        sboxes[i_local, c] = [0, 0, W, H]\n        cens[i_local, c] = [W // 2, H // 2]\n        return\n\n    img = vol[c]\n    max_val = img.max()\n    if max_val < 15:\n        sboxes[i_local, c] = [0, 0, W, H]\n        cens[i_local, c] = [W // 2, H // 2]\n        return\n\n    # Threshold tissue foreground\n    thresh = max(15, int(max_val * 0.12))\n    fg = img >= thresh\n\n    if fg.sum() < 40:\n        thresh = 10\n        fg = img >= thresh\n\n    if fg.sum() < 40:\n        sboxes[i_local, c] = [0, 0, W, H]\n        cens[i_local, c] = [W // 2, H // 2]\n        return\n\n    ys = np.where(fg.any(axis=1))[0]\n    xs = np.where(fg.any(axis=0))[0]\n\n    y0, y1 = int(ys[0]), int(ys[-1]) + 1\n    x0, x1 = int(xs[0]), int(xs[-1]) + 1\n\n    # Weighted centroid\n    w = img[fg].astype(np.float32)\n    w_sum = w.sum()\n    if w_sum > 0:\n        cx = float((x_grid[fg] * w).sum() / w_sum)\n        cy = float((y_grid[fg] * w).sum() / w_sum)\n    else:\n        cx = (x0 + x1) / 2.0\n        cy = (y0 + y1) / 2.0\n\n    sboxes[i_local, c] = [x0, y0, x1, y1]\n    cens[i_local, c] = [int(round(cx)), int(round(cy))]\n\n\ndef _roi_resize(tri, res):\n    """tri: (3,h,w) float32 [0,1] -> (3,res,res) float32. Mirrors train_knee StudyWindows._resize."""\n    t = torch.from_numpy(np.ascontiguousarray(tri))\n    if t.shape[-1] != res or t.shape[-2] != res:\n        t = F.interpolate(t[None], size=(res, res), mode="bilinear",\n                          align_corners=False)[0]\n    return t.numpy()\n\n\ndef _build_windows(vol, mask, res, clean_win=False, slots=None, k=None, roi=None, norm="none", zoom_px=0):\n    """Build the (K,3,res,res) float32 window batch for one study on CPU.\n\n    Returns (x, slot_ids, win_centers): x is norm-applied, slot_ids has one slot\n    per window, win_centers has one center per window (for TTA mirroring).\n\n    roi=None reproduces the legacy predict_study window loop exactly.\n    roi dict (mode/pad/overlap) reproduces train_knee StudyWindows.__getitem__\n    in eval mode: centers from pick_centers(..., slots=...), box from the CENTER\n    slice, compartment mode = 2 crops (left, right) per center, total capped at k.\n    """\n    valid = np.where(mask > 0)[0]\n    if len(valid) < 3:\n        valid = np.arange(min(3, MAXS))\n    kk = K_EVAL if k is None else int(k)\n    if roi is None:\n        centers = pick_centers(valid, kk, train=False, clean_windows=clean_win, mask=mask, max_d=MAXS, cap=True, slots=slots)\n        wins = []\n        slot_ids = []\n        for c in centers:\n            tri = vol[c - 1 : c + 2].astype(np.float32) / 255.0\n            tri_t = torch.from_numpy(tri)[None]\n            if res != IMG:\n                tri_t = F.interpolate(tri_t, size=(res, res), mode=\'bilinear\', align_corners=False)\n            wins.append(tri_t[0])\n            slot_ids.append(slot_of(c))\n            if zoom_px:   # second scale: centred close-up of the same window (train_knee StudyWindows parity)\n                oy, ox = (tri.shape[1] - zoom_px) // 2, (tri.shape[2] - zoom_px) // 2\n                z = torch.from_numpy(np.ascontiguousarray(tri[:, oy:oy + zoom_px, ox:ox + zoom_px]))[None]\n                wins.append(F.interpolate(z, size=(res, res), mode=\'bilinear\', align_corners=False)[0])\n                slot_ids.append(slot_of(c))\n        x = torch.stack(wins)\n        win_centers = [c for c in centers for _ in range(2 if zoom_px else 1)]\n    else:\n        mode = roi.get("mode", "compartment")\n        pad = float(roi.get("pad", 0.06))\n        overlap = float(roi.get("overlap", 0.12))\n        D = vol.shape[0]\n        H, W = vol.shape[-2], vol.shape[-1]\n        y_grid, x_grid = np.indices((H, W), dtype=np.float32)\n        sboxes = np.zeros((1, D, 4), dtype=np.int16)\n        cens = np.zeros((1, D, 2), dtype=np.int16)\n        for c in range(D):\n            _roi_slice_box(vol, mask, c, sboxes, cens, 0, W, H, x_grid, y_grid)\n        compart = (mode == "compartment")\n        n_centers = (kk + 1) // 2 if compart else kk\n        centers = pick_centers(valid, n_centers, train=False, clean_windows=clean_win, mask=mask, max_d=D, slots=slots)\n        wins = []\n        slot_ids = []\n        win_centers = []\n        for c in centers:\n            c = max(1, min(c, D - 2))\n            tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0\n            sq_mode = "tight" if compart else mode\n            sq = _roi_square_box(tuple(int(v) for v in sboxes[0, c]), W=W, H=H,\n                                 pad=pad, mode=sq_mode,\n                                 centroid=tuple(float(v) for v in cens[0, c]))\n            if compart:\n                left, right = _roi_compartments(sq, overlap=overlap)\n                for box in (left, right):\n                    x0, y0, x1, y1 = box\n                    wins.append(torch.from_numpy(_roi_resize(tri[:, y0:y1, x0:x1], res)))\n                    slot_ids.append(slot_of(c))\n                    win_centers.append(c)\n            else:\n                x0, y0, x1, y1 = sq\n                wins.append(torch.from_numpy(_roi_resize(tri[:, y0:y1, x0:x1], res)))\n                slot_ids.append(slot_of(c))\n                win_centers.append(c)\n        if len(wins) > kk:\n            wins = wins[:kk]\n            slot_ids = slot_ids[:kk]\n            win_centers = win_centers[:kk]\n        x = torch.stack(wins)\n    x = apply_norm(x, norm)\n    return x, slot_ids, win_centers\n\n\ndef check_fallback_guard(fallback_count: int, total_studies: int, max_rate: float = 0.03):\n    """Enforces strict fallback limit (< 3% of studies).\n\n    Raises RuntimeError if fallbacks exceed 3% of studies.\n    """\n    if total_studies <= 0:\n        return\n    rate = fallback_count / float(total_studies)\n    if rate > max_rate:\n        raise RuntimeError(\n            f"FATAL GUARD: fallback count {fallback_count}/{total_studies} "\n            f"({rate:.2%}) exceeded allowed limit of {max_rate:.1%}!"\n        )\n\n\ndef mirror_tri(tri_t, cs):\n    is_sag = torch.tensor([_s_map[c] in SAGITTAL_SLOTS for c in cs], device=tri_t.device).view(-1, 1, 1, 1)\n    return torch.where(is_sag, tri_t.flip(1), tri_t.flip(-1))\n\n\ndef _study_inputs(model, vol, mask, res, dev, side=0.0):\n    """Windows of one study for `model`\'s layout, on `dev`: (x, slots_t, win_centers)."""\n    roi_cfg = None\n    if getattr(model, "roi", False):\n        roi_cfg = {"mode": getattr(model, "roi_mode", "compartment"),\n                   "pad": float(getattr(model, "roi_pad", 0.06)),\n                   "overlap": float(getattr(model, "roi_overlap", 0.12))}\n    x, slot_ids, win_centers = _build_windows(\n        vol, mask, res,\n        clean_win=getattr(model, "clean_windows", False),\n        slots=getattr(model, "slots", None),\n        k=K_EVAL, roi=roi_cfg,\n        norm=getattr(model, "input_norm", "none"),\n        zoom_px=getattr(model, "zoom_px", 0))\n    if getattr(model, "seq_head", False):\n        assert all(b >= a for a, b in zip(win_centers, win_centers[1:])), \\\n            "seq_head requires ascending window centers"\n    x = x.to(dev, non_blocking=True)\n    if getattr(model, "pos_proj", None) is not None:\n        slots_t = pack_slot_pos(slot_ids, [window_pos(c, mask, side, False) for c in win_centers]).to(dev)[None]\n    else:\n        slots_t = torch.tensor(slot_ids, dtype=torch.long, device=dev)[None] if getattr(model, \'has_slot_emb\', False) else None\n    return x, slots_t, win_centers\n\n\ndef _probs(model, x, slots_t, win_centers, dev, use_tta):\n    autocast_on = str(dev).startswith(\'cuda\') and torch.cuda.is_available()\n    with torch.autocast(\'cuda\', dtype=torch.float16, enabled=autocast_on):\n        p_plain = torch.sigmoid(model(x[None], slots=slots_t).float())[0]\n        if use_tta:\n            x_m = mirror_tri(x, win_centers)\n            slots_m = slots_t\n            if slots_t is not None and slots_t.is_floating_point():   # mirrored knee: sagittal position flips\n                sid_ = slots_t.floor(); p_ = (slots_t - sid_) / 0.4995 - 1.0\n                slots_m = sid_ + (torch.where(sid_ < 2, -p_, p_) + 1.0) * 0.4995\n            p_mirr = torch.sigmoid(model(x_m[None], slots=slots_m).float())[0]\n            p_final = 0.5 * (p_plain + p_mirr)\n        else:\n            p_final = p_plain\n    return p_final.detach().cpu().numpy()\n\n\n@torch.no_grad()\ndef predict_study(model, vol, mask, res, dev, use_tta=False, side=0.0):\n    t_m0 = time.time()\n    x, slots_t, win_centers = _study_inputs(model, vol, mask, res, dev, side)\n    p_np = _probs(model, x, slots_t, win_centers, dev, use_tta)\n    return p_np, time.time() - t_m0\n\n\n@torch.no_grad()\ndef predict_study_multi(models, vol, mask, res, dev, use_tta=False, side=0.0):\n    """Mean probability of same-layout models (check_same_layout); the windows are built once."""\n    t_m0 = time.time()\n    x, slots_t, win_centers = _study_inputs(models[0], vol, mask, res, dev, side)\n    p_np = np.mean([_probs(m, x, slots_t, win_centers, dev, use_tta) for m in models], axis=0)\n    return p_np, time.time() - t_m0\n\n\n# ============================================================================\n# Path Discovery & Checkpoints\n# ============================================================================\ndef find_comp_source():\n    cands = [\n        os.environ.get("RSNA_COMP_ROOT", ""),\n        "/kaggle/input/rsna-knee-abnormality-detection",\n        "/kaggle/input/competitions/rsna-knee-abnormality-detection"\n    ]\n    for c in cands:\n        if c and os.path.exists(os.path.join(c, "train.csv")) and os.path.exists(os.path.join(c, "train_series")):\n            return Path(c)\n    for root, dirs, files in os.walk("/kaggle/input"):\n        if "train.csv" in files and "train_series" in dirs:\n            return Path(root)\n    return None\n\n\ndef find_corpus_source():\n    cands = [\n        "/kaggle/input/knee-raptor-corpus",\n        "/kaggle/input/datasets/dreaddevelopment/knee-raptor-corpus",\n        "ds/corpus1",\n        "/home/s05814802723/projects/knee/ds/corpus1"\n    ]\n    for c in cands:\n        if os.path.exists(os.path.join(c, "all_vols.npy")):\n            return Path(c)\n    for root, dirs, files in os.walk("/kaggle/input"):\n        if "all_vols.npy" in files:\n            return Path(root)\n    return None\n\n\ndef find_test_data():\n    comp_roots = []\n    if "RSNA_COMP_ROOT" in os.environ and os.path.exists(os.environ["RSNA_COMP_ROOT"]):\n        comp_roots.append(os.environ["RSNA_COMP_ROOT"])\n    comp_roots.extend([\n        "/kaggle/input/rsna-knee-abnormality-detection",\n        "/kaggle/input/competitions/rsna-knee-abnormality-detection"\n    ])\n    for r in comp_roots:\n        if not r or not os.path.exists(r):\n            continue\n        test_series_csv = os.path.join(r, "test_series.csv")\n        sample_sub_csv = os.path.join(r, "sample_submission.csv")\n        if not os.path.exists(test_series_csv):\n            continue\n        for cand in ["test_series", "test", "test_images"]:\n            p = os.path.join(r, cand)\n            if os.path.exists(p) and os.path.isdir(p):\n                print(f"[data] Found test image directory: {p}", flush=True)\n                return p, test_series_csv, sample_sub_csv\n    raise FileNotFoundError(f"Could not find competition test data. Checked: {comp_roots}")\n\n\nLAYOUT_KEYS = ("arch", "res", "norm", "clean_windows", "slot_emb", "slots", "slot_sizes", "seq_head",\n               "roi", "roi_mode", "roi_pad", "roi_overlap", "zoom_mm")\n\n\ndef check_same_layout(cks):\n    """Checkpoints scored on one decode pass must share the window layout; a mixed set would\n    silently feed one model another model\'s inputs."""\n    ref = {k: cks[0].get(k) for k in LAYOUT_KEYS}\n    for i, ck in enumerate(cks[1:], 1):\n        for k in LAYOUT_KEYS:\n            if ck.get(k) != ref[k]:\n                raise ValueError(f"checkpoint {i} differs in {k!r}: {ck.get(k)!r} != {ref[k]!r}")\n    return ref\n\n\ndef find_checkpoints():\n    """RAPTOR_CKPTS (comma list) scores several same-layout models on one decode pass."""\n    multi = os.environ.get("RAPTOR_CKPTS")\n    return [p for p in multi.split(",") if p] if multi else [find_checkpoint(prefer_swa=True)]\n\n\ndef find_checkpoint(prefer_swa=True):\n    if os.environ.get("RAPTOR_CKPT"):  # base954_fleet: pick the seed from the notebook\n        return os.environ["RAPTOR_CKPT"]\n    direct_candidates = [\n        "/kaggle/input/knee-our-models/raptor_ft_v6_swa.pt",\n        "/kaggle/input/datasets/shivoo29/knee-our-models/raptor_ft_v6_swa.pt",\n        "raptor_ft_v6_swa.pt",\n        "/kaggle/input/knee-our-models/raptor_ft_v6.pt",\n        "/kaggle/input/datasets/shivoo29/knee-our-models/raptor_ft_v6.pt",\n        "raptor_ft_v6.pt",\n        "/kaggle/input/knee-our-models/raptor_ft_v5s2_swa.pt",\n        "/kaggle/input/datasets/shivoo29/knee-our-models/raptor_ft_v5s2_swa.pt",\n        "raptor_ft_v5s2_swa.pt"\n    ]\n    for p in direct_candidates:\n        if os.path.exists(p):\n            print(f"[ckpt] Found direct checkpoint: {p}", flush=True)\n            return p\n\n    for root in ["/kaggle/input/knee-our-models", "/kaggle/input/datasets/shivoo29/knee-our-models", "."]:\n        if not os.path.exists(root): continue\n        target_fnames = ["raptor_ft_v6_swa.pt", "raptor_ft_v6.pt", "raptor_ft_v5s2_swa.pt", "raptor_ft_v5s2.pt"]\n        for fname in target_fnames:\n            p = os.path.join(root, fname)\n            if os.path.exists(p):\n                print(f"[ckpt] Selected checkpoint: {p}", flush=True)\n                return p\n    raise FileNotFoundError("Could not find model checkpoint in direct paths")\n\n\n# ============================================================================\n# Stage-by-Stage Corpus Parity Auditor (~20 Studies)\n# ============================================================================\ndef audit_corpus_parity(corpus_dir, comp_src, model, res, device, n_check=20):\n    print("\\n" + "=" * 80)\n    print(f"STAGE-BY-STAGE PARITY AUDIT: LIVE DICOM vs ALL_VOLS.NPY ({n_check} Studies)")\n    print("=" * 80)\n\n    c_ids = np.load(corpus_dir / "all_ids.npy", allow_pickle=True).astype(str)\n    c_vols = np.load(corpus_dir / "all_vols.npy", mmap_mode="r")\n    c_masks = np.load(corpus_dir / "all_masks.npy")\n    id2row = {sid: i for i, sid in enumerate(c_ids)}\n\n    ser_csv = comp_src / "train_series.csv"\n    ser_df = pd.read_csv(ser_csv, dtype=str)\n    ser_records = {}\n    for uid, g in ser_df.groupby("StudyInstanceUID"):\n        ser_records[uid] = g.to_dict("records")\n\n    tr = pd.read_csv(comp_src / "train.csv", dtype={"StudyInstanceUID": str})\n    gold_ids = [s for s in tr[tr.ACL.notna()].StudyInstanceUID if s in id2row][:n_check]\n\n    io_pool = ThreadPoolExecutor(max_workers=8)\n    train_series_dir = os.path.join(comp_src, "train_series")\n\n    diff_records = []\n    prob_diffs = []\n\n    for sid in gold_ids:\n        row = id2row[sid]\n        v_corpus = c_vols[row]\n        m_corpus = c_masks[row]\n\n        v_dicom, m_dicom, timings = build_study_optimized(sid, ser_records, train_series_dir, io_pool)\n\n        # Stage metrics\n        max_abs_diff = int(np.abs(v_corpus.astype(int) - v_dicom.astype(int)).max())\n        mean_abs_diff = float(np.abs(v_corpus.astype(float) - v_dicom.astype(float)).mean())\n        mask_match = bool(np.array_equal(m_corpus, m_dicom))\n\n        # Model output difference on both tensors\n        p_corp, _ = predict_study(model, v_corpus, m_corpus, res, device, use_tta=False)\n        p_dcm, _ = predict_study(model, v_dicom, m_dicom, res, device, use_tta=False, side=timings.get("side", 0.0))\n        max_p_diff = float(np.abs(p_corp - p_dcm).max())\n        cos_sim = float(np.dot(p_corp, p_dcm) / (np.linalg.norm(p_corp) * np.linalg.norm(p_dcm) + 1e-8))\n\n        diff_records.append({\n            "sid": sid[:12] + "...",\n            "max_px_diff": max_abs_diff,\n            "mean_px_diff": mean_abs_diff,\n            "mask_match": mask_match,\n            "max_prob_diff": max_p_diff,\n            "cos_sim": cos_sim\n        })\n        prob_diffs.append(max_p_diff)\n\n    io_pool.shutdown(wait=False)\n\n    df_report = pd.DataFrame(diff_records)\n    print(df_report.to_string(index=False))\n    print("-" * 80)\n    avg_max_p = np.mean(prob_diffs)\n    print(f"Parity Summary: Mean Max Prob Diff: {avg_max_p:.4f} | Mean Cosine Sim: {np.mean([r[\'cos_sim\'] for r in diff_records]):.5f}")\n    if avg_max_p < 0.05:\n        print("[audit] PARITY VERIFIED! DICOM path reproduces corpus tensor representations with high fidelity!")\n    else:\n        print(f"[audit] NOTICE: Slight numerical variance observed ({avg_max_p:.4f} prob diff), verifying on Gold-58 macro-AUC...")\n    print("=" * 80 + "\\n")\n\n\n# ============================================================================\n# Visible-Data Teacher Probe Setup & AUC Evaluation\n# ============================================================================\ndef setup_teacher_probe(comp_src, work_dir="/kaggle/working"):\n    fake = Path(work_dir) / "fakecomp"\n    fake.mkdir(parents=True, exist_ok=True)\n    tr = pd.read_csv(comp_src / "train.csv", dtype={"StudyInstanceUID": str})\n    gold_ids = tr[tr.ACL.notna()].StudyInstanceUID\n    n_sample = min(300, len(tr[tr.ACL.isna()]))\n    sampled_ids = tr[tr.ACL.isna()].StudyInstanceUID.sample(n_sample, random_state=0)\n    all_probe_ids = pd.concat([gold_ids, sampled_ids]).reset_index(drop=True)\n\n    # 1. test.csv\n    pd.DataFrame({"StudyInstanceUID": all_probe_ids}).to_csv(fake / "test.csv", index=False)\n\n    # 2. test_series.csv\n    ts = pd.read_csv(comp_src / "train_series.csv", dtype=str)\n    ts[ts.StudyInstanceUID.isin(all_probe_ids)].to_csv(fake / "test_series.csv", index=False)\n\n    # 3. sample_submission.csv\n    cols = ["StudyInstanceUID"] + LAB\n    pd.DataFrame({c: (all_probe_ids.values if c == "StudyInstanceUID" else 0.5) for c in cols}).to_csv(fake / "sample_submission.csv", index=False)\n\n    # 4. test_series symlink pointing to train_series\n    target_series = fake / "test_series"\n    if not target_series.exists():\n        src_train_series = comp_src / "train_series"\n        target_series.symlink_to(src_train_series)\n\n    print(f"[probe] Built fakecomp at {fake}: {len(gold_ids)} gold + {n_sample} train studies = {len(all_probe_ids)} total", flush=True)\n    return fake, gold_ids.tolist()\n\n\ndef evaluate_gold_auc(sub_df, train_csv_path):\n    from sklearn.metrics import roc_auc_score\n    tr = pd.read_csv(train_csv_path, dtype={\'StudyInstanceUID\': str})\n    gold_df = tr[tr.ACL.notna()].copy()\n    gold_ids = set(gold_df.StudyInstanceUID)\n    matched = sub_df[sub_df.StudyInstanceUID.isin(gold_ids)].copy()\n\n    if len(matched) < len(gold_ids):\n        print(f"[eval] Warning: Only matched {len(matched)} / {len(gold_ids)} gold studies", flush=True)\n    if len(matched) == 0:\n        print("[eval] No gold studies matched in submission. Skipping AUC evaluation.", flush=True)\n        return None\n\n    # Align exactly by StudyInstanceUID\n    gold_df = gold_df[gold_df.StudyInstanceUID.isin(matched.StudyInstanceUID)].copy()\n    gold_df = gold_df.sort_values("StudyInstanceUID").reset_index(drop=True)\n    matched = matched.sort_values("StudyInstanceUID").reset_index(drop=True)\n\n    aucs = {}\n    print("\\n" + "=" * 80)\n    print(f"VISIBLE GOLD-58 VALIDATION RESULTS (K_EVAL={K_EVAL})")\n    print("=" * 80)\n    for col in LAB:\n        y_true = gold_df[col].values.astype(float)\n        y_pred = matched[col].values.astype(float)\n        try:\n            score = roc_auc_score(y_true, y_pred)\n            aucs[col] = score\n            print(f"  {col:<20}: {score:.4f}")\n        except Exception as e:\n            print(f"  {col:<20}: ERR ({e})")\n    macro_auc = float(np.mean(list(aucs.values()))) if aucs else 0.0\n    print("-" * 80)\n    print(f"  MACRO-AUC (Gold 58): {macro_auc:.4f}")\n    print("=" * 80 + "\\n")\n    return macro_auc\n\n\n# ============================================================================\n# Multi-GPU Pipeline Executor\n# ============================================================================\ndef run_pipeline(test_dir, test_series_csv, sample_sub_csv, devices, models, res, is_probe=False):\n    t_start = time.time()\n    tag = "PROBE" if is_probe else "PRODUCTION"\n    print(f"\\n[{tag}] Initializing pipeline run...")\n    print(f"[{tag}] Image dir: {test_dir} | Series CSV: {test_series_csv}")\n\n    ser_df = pd.read_csv(test_series_csv)\n    ser_df["StudyInstanceUID"] = ser_df["StudyInstanceUID"].astype(str)\n    ser_records = {}\n    for uid, g in ser_df.groupby("StudyInstanceUID"):\n        ser_records[uid] = g.to_dict("records")\n\n    sample_sub = pd.read_csv(sample_sub_csv)\n    test_ids = sample_sub["StudyInstanceUID"].astype(str).tolist()\n    n_studies = len(test_ids)\n    print(f"[{tag}] Scoring {n_studies} studies with K_EVAL={K_EVAL}...", flush=True)\n\n    # Shared queues and thread pools\n    io_pool = ThreadPoolExecutor(max_workers=12)\n    work_queue = queue.Queue(maxsize=8)\n    results = [None] * n_studies\n    profile_records = []\n    fallback_count = 0\n    records_lock = threading.Lock()\n    t_steady_start = time.time()\n\n    def gpu_worker(worker_idx, dev, model):\n        nonlocal fallback_count\n        while True:\n            item = work_queue.get()\n            if item is None:\n                work_queue.task_done()\n                break\n            idx, sid, vol, mask, timings = item\n            is_fallback = False\n\n            # Check if volume has zero decoded slices\n            if timings.get("decoded_slices", 0) == 0:\n                is_fallback = True\n                with records_lock:\n                    fallback_count += 1\n                record_exception(f"gpu_worker_{worker_idx}_zero_slices", sid, RuntimeError("Zero valid slices decoded for study!"))\n\n            try:\n                preds, t_model = predict_study_multi(model, vol, mask, res, dev, use_tta=USE_TTA,\n                                                     side=timings.get("side", 0.0))\n                timings["model"] = t_model\n                timings["worker"] = worker_idx\n            except Exception as e:\n                if not is_fallback:\n                    with records_lock:\n                        fallback_count += 1\n                record_exception(f"gpu_worker_{worker_idx}_predict", sid, e)\n                preds = np.full(12, 0.5, dtype=np.float32)\n                timings["model"] = 0.0\n                timings["worker"] = worker_idx\n\n            row = {"StudyInstanceUID": sid}\n            for j, col in enumerate(LAB):\n                row[col] = float(preds[j])\n            results[idx] = row\n\n            with records_lock:\n                profile_records.append(timings)\n                done_cnt = len(profile_records)\n                if done_cnt % 25 == 0 or done_cnt == n_studies:\n                    cur_steady = time.time() - t_steady_start\n                    print(f"[{tag}] [{done_cnt}/{n_studies}] {cur_steady / done_cnt:.3f}s/study (steady) | {cur_steady:.1f}s elapsed", flush=True)\n            work_queue.task_done()\n\n    # Launch GPU consumer threads\n    consumer_threads = []\n    for w_idx, (dev, m) in enumerate(zip(devices, models)):\n        t = threading.Thread(target=gpu_worker, args=(w_idx, dev, m), daemon=True)\n        t.start()\n        consumer_threads.append(t)\n\n    # Producer: load and preprocess studies in background\n    for idx, sid in enumerate(test_ids):\n        try:\n            vol, mask, timings = build_study_optimized(sid, ser_records, test_dir, io_pool)\n        except Exception as e:\n            fallback_count += 1\n            record_exception("producer_build_study", sid, e)\n            vol = np.zeros((MAXS, IMG, IMG), np.uint8)\n            mask = np.zeros(MAXS, np.uint8)\n            timings = {"header": 0.0, "pixel": 0.0, "prep": 0.0, "total_io": 0.0, "decoded_slices": 0, "total_requested": 0}\n        work_queue.put((idx, sid, vol, mask, timings))\n\n    # Wait for queue completion\n    work_queue.join()\n    for _ in devices:\n        work_queue.put(None)\n    for t in consumer_threads:\n        t.join()\n    io_pool.shutdown(wait=False)\n\n    # Assemble and validate submission DataFrame\n    for idx, r in enumerate(results):\n        if r is None:\n            sid = test_ids[idx]\n            print(f"[FATAL ERROR] results[{idx}] for study {sid} is missing!", flush=True)\n            results[idx] = {"StudyInstanceUID": sid, **{col: 0.5 for col in LAB}}\n\n    sub = pd.DataFrame(results)\n    assert len(sub) == n_studies, f"Row count mismatch: {len(sub)} vs {n_studies}"\n    assert list(sub["StudyInstanceUID"]) == test_ids, "Row ID ordering drift!"\n    sub = sub[["StudyInstanceUID"] + LAB]\n\n    # Metrics\n    steady_time = time.time() - t_steady_start\n    avg_steady_per_study = steady_time / max(1, n_studies)\n    mean_header = np.mean([p["header"] for p in profile_records]) if profile_records else 0\n    mean_pixel = np.mean([p["pixel"] for p in profile_records]) if profile_records else 0\n    mean_prep = np.mean([p["prep"] for p in profile_records]) if profile_records else 0\n    mean_model = np.mean([p.get("model", 0) for p in profile_records]) if profile_records else 0\n    total_decoded_slices = sum(p.get("decoded_slices", 0) for p in profile_records)\n\n    print("\\n" + "-" * 80)\n    print(f"[{tag}] PERFORMANCE PROFILE ({n_studies} Studies)")\n    print("-" * 80)\n    print(f"  • Steady Throughput:       {avg_steady_per_study:.3f} s / study")\n    print(f"  • Total Decoded Slices:    {total_decoded_slices} slices")\n    print(f"  • Header & IPP Sorting:    {mean_header*1000:6.1f} ms")\n    print(f"  • Pixel Decompression:     {mean_pixel*1000:6.1f} ms")\n    print(f"  • Normalization & Crop:    {mean_prep*1000:6.1f} ms")\n    print(f"  • Model Forward:           {mean_model*1000:6.1f} ms")\n    print(f"  • Fallbacks Encountered:   {fallback_count} / {n_studies}")\n    print("-" * 80 + "\\n")\n\n    return sub, profile_records, fallback_count\n\n\n# ============================================================================\n# Main Entry Point\n# ============================================================================\ndef main():\n    global SLOTS, MAXS, K_EVAL, SLOT_SIZES, SLOT_BOUNDS, _b_slots, _s_map, PICK_SPAN\n    t_main_start = time.time()\n    print("=" * 80)\n    print(f"RSNA Knee Inference Pipeline | Mode: {\'FULL (TTA)\' if USE_TTA else \'FAST (No-TTA)\'} | K_EVAL={K_EVAL}")\n    print("=" * 80)\n\n    # Detect hardware\n    num_gpus = torch.cuda.device_count()\n    if num_gpus > 1:\n        devices = [f"cuda:{i}" for i in range(num_gpus)]\n        print(f"[devices] Dual-T4 Multi-GPU enabled: {devices}", flush=True)\n    elif num_gpus == 1:\n        devices = ["cuda:0"]\n        print(f"[devices] Single GPU enabled: {devices[0]}", flush=True)\n    else:\n        devices = ["cpu"]\n        print(f"[devices] CPU only (warning: slow)", flush=True)\n\n    ckpt_paths = find_checkpoints()\n    cks = [torch.load(p, map_location="cpu", weights_only=False) for p in ckpt_paths]\n    check_same_layout(cks)\n    ck, ckpt_path = cks[0], ",".join(ckpt_paths)\n    arch = ck.get("arch", "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k")\n    res = int(ck.get("res", 384))\n    norm = ck.get("norm", "none")\n    clean_win = bool(ck.get("clean_windows", False))\n    has_slot = bool(ck.get("slot_emb", False))\n    spec_slots = tuple(int(v) for v in ck["slots"].split(",")) if ck.get("slots") else None\n    # Slot layout follows the checkpoint (any 5-slot corpus, e.g. 192 slices);\n    # old checkpoints without the key keep the 96 defaults bit-for-bit.\n    ck_sizes = [int(v) for v in str(ck.get("slot_sizes", "26,22,18,12,18")).split(",")]\n    PICK_SPAN = (0.02, 0.98) if ck.get("slot_sizes") else (0.06, 0.94)   # old checkpoints: unchanged inference\n    assert len(ck_sizes) == 5 and all(v > 0 for v in ck_sizes), f"bad slot_sizes: {ck.get(\'slot_sizes\')}"\n    SLOT_SIZES = list(ck_sizes)\n    SLOTS = [(pl, fl, k) for (pl, fl, _), k in zip(SLOTS, SLOT_SIZES)]\n    MAXS = sum(SLOT_SIZES)\n    K_EVAL = MAXS - 2\n    SLOT_BOUNDS = bounds_from_sizes(SLOT_SIZES)\n    _b_slots = np.cumsum([0] + SLOT_SIZES)\n    _s_map = np.searchsorted(_b_slots, np.arange(MAXS), side=\'right\') - 1\n    print(f"[model] Checkpoint: {ckpt_path} | Arch: {arch} | Res: {res} | clean_win={clean_win} | slot_emb={has_slot} | slots={SLOT_SIZES} | MAXS={MAXS} K_EVAL={K_EVAL}", flush=True)\n\n    # Initialize model replica per device\n    models = []\n    sample_bb = build_backbone(arch, pretrained=False)\n    f_dim = sample_bb.num_features\n    del sample_bb\n\n    for dev in devices:\n        group = []\n        for c in cks:\n            bb = build_backbone(arch, pretrained=False)\n            m = RaptorClassifier(bb, F_dim=f_dim, slot_emb=has_slot, seq_head=bool(ck.get("seq_head", False)),\n                                 pos_emb="pos_proj.weight" in c["model"]).to(dev)\n            m.load_state_dict(c["model"])\n            m.win_chunk = WIN_CHUNK  # chunked eval as before (class default 0 = whole study)\n            m.input_norm = norm\n            m.clean_windows = clean_win\n            m.has_slot_emb = has_slot\n            m.slots = spec_slots\n            m.roi = bool(ck.get("roi", False))\n            m.roi_mode = ck.get("roi_mode", "compartment")\n            m.roi_pad = float(ck.get("roi_pad", 0.06))\n            m.roi_overlap = float(ck.get("roi_overlap", 0.12))\n            m.zoom_px = zoom_px(float(ck.get("zoom_mm", 0.0)))\n            m.seq_tracks = 2 if m.zoom_px else 1\n            m.eval()\n            group.append(m)\n        models.append(group)\n    print(f"[model] {len(cks)} checkpoint(s) per device on one decode pass", flush=True)\n\n    # Pre-warm GPU CUDA runtime context and cuDNN autotuner\n    print("[warmup] Pre-warming CUDA runtime & cuDNN autotuner across devices...", flush=True)\n    for dev, group in zip(devices, models):\n        for m in group:\n            dummy_chunk = torch.zeros((WIN_CHUNK, 3, res, res), dtype=torch.float32, device=dev)\n            dummy_in = dummy_chunk[None]\n            dummy_slots = torch.zeros((1, WIN_CHUNK), dtype=torch.long, device=dev) if getattr(m, \'has_slot_emb\', False) else None\n            with torch.no_grad(), torch.autocast(\'cuda\', dtype=torch.float16, enabled=torch.cuda.is_available()):\n                _ = m(dummy_in, slots=dummy_slots)\n        if torch.cuda.is_available():\n            torch.cuda.synchronize(dev)\n    t_startup = time.time() - t_main_start\n    print(f"[startup] Fixed setup & GPU warmup completed in {t_startup:.2f}s", flush=True)\n\n    # Identify real competition test inputs\n    real_test_dir, real_test_series_csv, real_sample_sub_csv = find_test_data()\n    real_sample_sub = pd.read_csv(real_sample_sub_csv)\n    n_real = len(real_sample_sub)\n    print(f"[data] Detected real test set: {n_real} studies from {real_test_dir}", flush=True)\n\n    # Check for competition source datasets to enable Teacher Probe & Parity Auditor\n    comp_src = find_comp_source()\n    corpus_src = find_corpus_source()\n\n    # STAGE 0: CORPUS PARITY AUDIT (if corpus dataset is attached and visible data available)\n    if comp_src is not None and corpus_src is not None and n_real <= 10:\n        try:\n            audit_corpus_parity(corpus_src, comp_src, models[0][0], res, devices[0], n_check=20)\n        except Exception as e:\n            print(f"[audit] Corpus parity audit exception (non-fatal): {e}", flush=True)\n            traceback.print_exc()\n\n    # STAGE 1: TEACHER PROBE VALIDATION ON VISIBLE DATA\n    should_run_probe = (comp_src is not None) and ((n_real <= 10) or (os.environ.get("RUN_PROBE", "0") == "1"))\n\n    if should_run_probe:\n        print("\\n" + "=" * 80)\n        print("STAGE 1: RUNNING TEACHER PROBE VALIDATION ON VISIBLE DATA")\n        print("=" * 80)\n        probe_root, gold_ids = setup_teacher_probe(comp_src)\n        probe_test_dir = os.path.join(probe_root, "test_series")\n        probe_test_series_csv = os.path.join(probe_root, "test_series.csv")\n        probe_sample_sub_csv = os.path.join(probe_root, "sample_submission.csv")\n\n        probe_sub, probe_records, probe_fallbacks = run_pipeline(\n            probe_test_dir, probe_test_series_csv, probe_sample_sub_csv,\n            devices, models, res, is_probe=True\n        )\n\n        # AUDIT 1: Assert IDs match test.csv and columns match sample_submission.csv\n        probe_test_csv = os.path.join(probe_root, "test.csv")\n        expected_test = pd.read_csv(probe_test_csv, dtype={"StudyInstanceUID": str})\n        assert list(probe_sub["StudyInstanceUID"]) == list(expected_test["StudyInstanceUID"]), \\\n            "FATAL: Probe output StudyInstanceUID order does NOT match test.csv!"\n        assert list(probe_sub.columns) == ["StudyInstanceUID"] + LAB, \\\n            "FATAL: Probe output columns do NOT match sample_submission.csv!"\n\n        # AUDIT 2: Hard guard on fallback count\n        probe_n = len(probe_sub)\n        fallback_rate = probe_fallbacks / max(1, probe_n)\n        print(f"[guard] Probe Fallback Rate: {probe_fallbacks}/{probe_n} ({fallback_rate:.2%})", flush=True)\n        if fallback_rate > 0.05:\n            raise RuntimeError(f"FATAL GUARD TRIGGERED: Probe fallback rate {fallback_rate:.2%} > 5% allowed limit!")\n\n        # AUDIT 3: Validate Gold-58 macro-AUC\n        macro_auc = evaluate_gold_auc(probe_sub, os.path.join(comp_src, "train.csv"))\n        if macro_auc is not None:\n            if macro_auc < 0.85:\n                raise RuntimeError(f"FATAL GUARD TRIGGERED: Gold-58 Macro-AUC is {macro_auc:.4f} < 0.8500! Model predictions are broken/constant!")\n            print(f"[audit] SUCCESS! Visible Gold-58 Macro-AUC: {macro_auc:.4f} (passed >= 0.8500 check)", flush=True)\n    else:\n        print(f"[info] Skipping Teacher Probe (comp_src={comp_src}, n_real={n_real}, RUN_PROBE={os.environ.get(\'RUN_PROBE\')})", flush=True)\n\n    # STAGE 2: PRODUCTION INFERENCE ON REAL TEST SET\n    print("\\n" + "=" * 80)\n    print(f"STAGE 2: SCORING REAL TEST SET ({n_real} Studies)")\n    print("=" * 80)\n    final_sub, final_records, final_fallbacks = run_pipeline(\n        real_test_dir, real_test_series_csv, real_sample_sub_csv,\n        devices, models, res, is_probe=False\n    )\n\n    # Final submission verification\n    assert list(final_sub["StudyInstanceUID"]) == list(real_sample_sub["StudyInstanceUID"]), \\\n        "FATAL: Final submission StudyInstanceUID order does NOT match sample_submission.csv!"\n    assert list(final_sub.columns) == list(real_sample_sub.columns), \\\n        "FATAL: Final submission columns do NOT match sample_submission.csv!"\n    assert not final_sub.isna().any().any(), "FATAL: Final submission contains NaNs!"\n\n    if n_real > 10:   # the 3-study commit run must not trip it\n        check_fallback_guard(final_fallbacks, n_real, max_rate=0.03)\n\n    final_sub.to_csv(os.environ.get("RAPTOR_OUTPUT_CSV", "submission.csv"), index=False)\n    print(f"\\n[complete] submission.csv saved successfully with shape {final_sub.shape} and verified valid values!", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


In [ ]:

# [own] our model families as one rank branch; each group = one fleet_v6 decode pass over the test set
import glob, os, subprocess, sys, time
import numpy as np, pandas as pd
W_OWN = 0.3
W_FIND = {}   # per-finding override of W_OWN
GROUPS = {'r2': ['raptor_ft_cnxt_r2_f0_swa.pt', 'raptor_ft_cnxt_r2_f1_swa.pt', 'raptor_ft_cnxt_r2_f2_swa.pt', 'raptor_ft_cnxt_r2_f3_swa.pt']}
STOP_AFTER_H = 7.0   # no group starts after this; skipped groups leave the rest unchanged
KILL_AT_H = 8.4      # a group still running at this notebook hour is killed (base kept), so 9 h is never hit
HAS_BASE = False

def find_one(name):
    hits = glob.glob('/kaggle/input/*/' + name) + glob.glob('/kaggle/input/datasets/*/*/' + name)
    assert hits, name + ' not attached'
    return hits[0]

Path('/tmp/fleet_v6.py').write_text(FLEET_V6)
outs = {}
for tag, names in GROUPS.items():
    h = (time.time() - NB_T0) / 3600
    if h > STOP_AFTER_H:
        print('OWN_SKIPPED %s at %.2f h' % (tag, h), flush=True)
        continue
    out = WORK / ('own_%s.csv' % tag)
    env = dict(os.environ, RSNA_COMP_ROOT=str(DATA), RAPTOR_OUTPUT_CSV=str(out),
               RAPTOR_CKPTS=','.join(find_one(x) for x in names))
    t = time.time()
    try:
        rc = subprocess.run([sys.executable, '-u', '/tmp/fleet_v6.py'], env=env, cwd='/tmp',
                            timeout=max(1.0, KILL_AT_H * 3600 - (t - NB_T0))).returncode
    except subprocess.TimeoutExpired:
        rc = 'timeout at %.2f h' % KILL_AT_H
    if rc != 0:
        if HAS_BASE:
            print('OWN_FAILED %s exit %s: group skipped, base kept' % (tag, rc), flush=True)
            continue
        raise RuntimeError('own group %s failed with exit code %s' % (tag, rc))
    print('OWN_DONE %s %.0fs' % (tag, time.time() - t), flush=True)
    outs[tag] = pd.read_csv(out, dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID').loc[test_ids, labels]

if not outs:
    assert HAS_BASE, 'no own group ran and there is no base submission'
    print('OWN_NONE: keeping the base submission', flush=True)
else:
    base = (pd.read_csv(WORK / 'submission.csv', dtype={'StudyInstanceUID': str})
              .set_index('StudyInstanceUID').loc[test_ids, labels]) if HAS_BASE else None
    final = np.zeros((n, len(labels)))
    rho = {}
    for j, c in enumerate(labels):
        own = np.mean([pct(o[c].values) for o in outs.values()], axis=0)
        if HAS_BASE:
            w = W_FIND.get(c, W_OWN)
            final[:, j] = (1 - w) * pct(base[c].values) + w * own
            if n >= 3:
                rho[c] = round(float(pd.Series(base[c].values).corr(pd.Series(own), method='spearman')), 4)
        else:
            final[:, j] = own
    sub = pd.DataFrame(final.astype(np.float32), columns=labels)
    sub.insert(0, 'StudyInstanceUID', test_ids)
    sub = sub[cols]
    assert sub['StudyInstanceUID'].tolist() == test_ids and np.isfinite(sub[labels].values).all()
    assert ((sub[labels].values >= 0) & (sub[labels].values <= 1)).all()
    sub.to_csv(WORK / 'submission.csv', index=False)
    print('OWN_BLEND_COMPLETE', json.dumps({'w_own': W_OWN if HAS_BASE else 1.0, 'w_find': W_FIND, 'groups': sorted(outs),
                                             'total_seconds': round(time.time() - NB_T0, 1),
                                             'spearman_base_vs_own': rho}), flush=True)
